# 04 — Feature Engineering

**Projet** : FakeNews Analyzer — DevComplex  
**Objectif** : TF-IDF Spark, features NLP (VADER sentiment), export pour Colab.

**Exécuter après** : `03_unification.ipynb`

---

In [ ]:
# ============================================================
# Fichier  : 04_feature_engineering.ipynb
# Rôle     : TF-IDF, embeddings, features NLP, export Colab
# Version  : V1 (local)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================

import sys, os
import numpy as np
sys.path.insert(0, '..')

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF

from spark_utils import get_spark_session, load_parquet, save_parquet

spark = get_spark_session(app_name='FakeNews-Features', memory='8g')
PROCESSED_DIR = '../09_data/processed'
MODELS_DIR    = '../04_models'
COLAB_DIR     = '../09_data/colab_exports'

os.makedirs(COLAB_DIR, exist_ok=True)
print('Session Spark démarrée')

## Section 1 — Chargement des données

In [ ]:
train_path = os.path.join(PROCESSED_DIR, 'train.parquet')
test_path  = os.path.join(PROCESSED_DIR, 'test.parquet')

train_df = load_parquet(spark, train_path)
test_df  = load_parquet(spark, test_path)

if train_df is None or test_df is None:
    raise FileNotFoundError('train.parquet ou test.parquet introuvable. Exécuter 03_unification.ipynb.')

print(f'Train : {train_df.count():,} lignes')
print(f'Test  : {test_df.count():,} lignes')
train_df.show(2, truncate=80)

## Section 2 — Pipeline TF-IDF Spark

In [ ]:
# Pipeline MLlib : Tokenizer → StopWordsRemover → HashingTF → IDF
tokenizer = RegexTokenizer(
    inputCol='clean_text', outputCol='tokens_raw',
    pattern=r'\W+', minTokenLength=3
)
remover = StopWordsRemover(
    inputCol='tokens_raw', outputCol='tokens',
    stopWords=StopWordsRemover.loadDefaultStopWords('english')
)
hashing_tf = HashingTF(
    inputCol='tokens', outputCol='tf_features',
    numFeatures=50_000    # Vocabulaire max
)
idf = IDF(
    inputCol='tf_features', outputCol='tfidf_features',
    minDocFreq=2          # Ignorer les termes rares (< 2 docs)
)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])

print('Entraînement du pipeline TF-IDF sur le train...')
tfidf_model = tfidf_pipeline.fit(train_df)

train_features = tfidf_model.transform(train_df)
test_features  = tfidf_model.transform(test_df)

print('✓ Pipeline TF-IDF entraîné')

In [ ]:
# Sauvegarder le modèle TF-IDF Spark
tfidf_path = os.path.join(MODELS_DIR, 'baseline', 'tfidf_vectorizer')
os.makedirs(tfidf_path, exist_ok=True)
tfidf_model.write().overwrite().save(tfidf_path)
print(f'✓ Modèle TF-IDF sauvegardé : {tfidf_path}')

## Section 3 — Features NLP (VADER Sentiment)

In [ ]:
# Features NLP additionnelles : sentiment VADER, longueur, ponctuation
def extract_nlp_features_udf(text: str):
    """Extrait les features NLP sur le texte original."""
    if not text:
        return (0, 0.0, 0.0, 0, 0)
    
    word_count = len(text.split())
    alpha_chars = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in alpha_chars if c.isupper()) / len(alpha_chars) if alpha_chars else 0.0
    exclamation_count = text.count('!')
    question_count = text.count('?')
    
    sentiment_score = 0.0
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        analyzer = SentimentIntensityAnalyzer()
        sentiment_score = analyzer.polarity_scores(text)['compound']
    except Exception:
        pass
    
    return (word_count, float(sentiment_score), float(capital_ratio), exclamation_count, question_count)

from pyspark.sql.types import StructType, StructField

nlp_schema = StructType([
    StructField('word_count', IntegerType()),
    StructField('sentiment_score', FloatType()),
    StructField('capital_ratio', FloatType()),
    StructField('exclamation_count', IntegerType()),
    StructField('question_count', IntegerType()),
])

nlp_udf = F.udf(extract_nlp_features_udf, nlp_schema)

train_with_nlp = train_df.withColumn('nlp', nlp_udf(F.col('clean_text'))).select(
    '*',
    F.col('nlp.word_count').alias('word_count'),
    F.col('nlp.sentiment_score').alias('sentiment_score'),
    F.col('nlp.capital_ratio').alias('capital_ratio'),
    F.col('nlp.exclamation_count').alias('exclamation_count'),
    F.col('nlp.question_count').alias('question_count'),
).drop('nlp')

print('✓ Features NLP calculées')
train_with_nlp.show(3, truncate=80)

## Section 4 — Export pour Google Colab

In [ ]:
import scipy.sparse as sp

def spark_vector_to_scipy(df, feature_col: str, label_col: str, n_features: int = 50_000):
    """Convertit les SparseVectors Spark en matrice scipy pour l'export Colab."""
    from pyspark.ml.linalg import SparseVector
    
    rows = df.select(feature_col, label_col).collect()
    labels = np.array([row[label_col] for row in rows])
    
    data_rows = []
    for row in rows:
        vec = row[feature_col]
        if hasattr(vec, 'toArray'):
            data_rows.append(vec.toArray())
        else:
            arr = np.zeros(n_features)
            if hasattr(vec, 'indices') and hasattr(vec, 'values'):
                arr[vec.indices] = vec.values
            data_rows.append(arr)
    
    matrix = sp.csr_matrix(np.vstack(data_rows))
    return matrix, labels

print('Export des features TF-IDF pour Colab...')
X_train, y_train = spark_vector_to_scipy(train_features, 'tfidf_features', 'label')
X_test,  y_test  = spark_vector_to_scipy(test_features,  'tfidf_features', 'label')

sp.save_npz(os.path.join(COLAB_DIR, 'train_features.npz'), X_train)
sp.save_npz(os.path.join(COLAB_DIR, 'test_features.npz'),  X_test)
np.save(os.path.join(COLAB_DIR, 'train_labels.npy'), y_train)
np.save(os.path.join(COLAB_DIR, 'test_labels.npy'),  y_test)

print(f'✓ Matrices TF-IDF exportées : {X_train.shape} train | {X_test.shape} test')

In [ ]:
# Exporter aussi les textes bruts pour DistilBERT
train_texts = train_df.select('clean_text', 'label').toPandas()
test_texts  = test_df.select('clean_text', 'label').toPandas()

train_texts.to_csv(os.path.join(COLAB_DIR, 'train_texts.csv'), index=False)
test_texts.to_csv(os.path.join(COLAB_DIR, 'test_texts.csv'),   index=False)

print(f'✓ Textes exportés : {len(train_texts):,} train | {len(test_texts):,} test')
print(f'\nFichiers dans {COLAB_DIR} :')
for f in os.listdir(COLAB_DIR):
    size = os.path.getsize(os.path.join(COLAB_DIR, f)) / 1024 / 1024
    print(f'  {f:<30} {size:.1f} MB')

spark.stop()
print('\n✓ Session Spark arrêtée')
print('Prochaine étape : exécuter 03_modeling/01_baseline_tfidf.ipynb')